# YOLOv8 학습 — 노트북에서 직접 실행
ml.g5.12xlarge (A10G x4, 24GB each) 에서 직접 학습. Run All.

In [ ]:
# ============================================================
# Cell 1: 환경 설정
# ============================================================
!pip install ultralytics==8.3.52 -q
!pip uninstall ray -y -q 2>/dev/null

# numpy trapz 패치 — DDP 서브프로세스에도 적용되도록 .pth 파일 설치
import site, os
pth_code = "import numpy; numpy.trapz = getattr(numpy, 'trapezoid', getattr(numpy, 'trapz', None))\n"
for sp in site.getsitepackages():
    pth_path = os.path.join(sp, 'numpy_trapz_fix.pth')
    with open(pth_path, 'w') as f:
        f.write(pth_code)
    print(f'numpy trapz 패치 설치: {pth_path}')

# 현재 프로세스에도 적용
import numpy as np
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid
    print(f'numpy {np.__version__}: trapz → trapezoid 패치 적용')

import torch
print(f'PyTorch: {torch.__version__}')
n_gpu = torch.cuda.device_count()
print(f'CUDA: {torch.cuda.is_available()}, GPU {n_gpu}개')
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {props.name} ({props.total_memory / 1e9:.1f}GB)')

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')
print('\n설정 완료!')

In [ ]:
# ============================================================
# Cell 2: 데이터 + 모델 준비
# ============================================================
import os, glob, subprocess, shutil, time

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
REGION = 'ap-northeast-2'
BASE = '/home/ec2-user/SageMaker/vindr-cxr'
YOLO_DIR = f'{BASE}/yolo_dataset'
TRAIN_DIR = f'{BASE}/yolov8_runs'

# 데이터 확인 (이미 로컬에 있으면 스킵)
data_yaml = f'{YOLO_DIR}/data.yaml'
if os.path.exists(data_yaml):
    print(f'데이터 이미 존재: {YOLO_DIR}')
else:
    print(f'S3에서 데이터 다운로드...')
    !aws s3 sync s3://{BUCKET}/vindr-cxr/processed/ {YOLO_DIR}/ --region {REGION} --only-show-errors

# data.yaml 경로 수정 (노트북용)
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val: images/val

nc: 14
names:
  0: Aortic_enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung_Opacity
  8: Nodule_Mass
  9: Other_lesion
  10: Pleural_effusion
  11: Pleural_thickening
  12: Pneumothorax
  13: Pulmonary_fibrosis
"""
with open(data_yaml, 'w') as f:
    f.write(yaml_content)
print(f'data.yaml 업데이트: path={YOLO_DIR}')

# 이미지 수 확인
for split in ['train', 'val']:
    d = f'{YOLO_DIR}/images/{split}'
    if os.path.isdir(d):
        print(f'  {split}: {len(os.listdir(d)):,}장')

# yolov8s.pt 준비
model_pt = '/home/ec2-user/SageMaker/yolov8s.pt'
if os.path.exists(model_pt) and os.path.getsize(model_pt) > 20_000_000:
    print(f'\n모델 이미 존재: {model_pt}')
else:
    # S3에서 가져오기
    ret = subprocess.run(
        f'aws s3 cp s3://{BUCKET}/models/yolov8s.pt {model_pt} --region {REGION} --only-show-errors',
        shell=True, capture_output=True, text=True
    )
    if ret.returncode == 0:
        print(f'\n모델 S3 다운로드: {model_pt}')
    else:
        # Ultralytics로 다운로드
        print('\n모델 다운로드 중...')
        from ultralytics import YOLO
        _ = YOLO('yolov8s.pt')
        if os.path.exists('yolov8s.pt'):
            shutil.move('yolov8s.pt', model_pt)
        print(f'모델: {model_pt}')

print(f'모델 크기: {os.path.getsize(model_pt)/1e6:.1f}MB')
print('\n준비 완료!')

In [ ]:
# ============================================================
# Cell 3: 학습 실행 (ml.g5.12xlarge — A10G x4)
# ============================================================
import numpy as np
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid

import os
os.environ['YOLO_OFFLINE'] = '1'

from ultralytics import YOLO
import torch, time

YOLO_DIR = '/home/ec2-user/SageMaker/vindr-cxr/yolo_dataset'
TRAIN_DIR = '/home/ec2-user/SageMaker/vindr-cxr/yolov8_runs'
model_pt = '/home/ec2-user/SageMaker/yolov8s.pt'
data_yaml = f'{YOLO_DIR}/data.yaml'

model = YOLO(model_pt)

n_gpu = torch.cuda.device_count()
device = list(range(n_gpu)) if n_gpu > 1 else 0
batch = 16 * max(n_gpu, 1)  # GPU당 16 → 4GPU면 64

print(f'모델 로드 완료: {model_pt}')
print(f'GPU: {n_gpu}개, device={device}, batch={batch}')

start = time.time()

results = model.train(
    data=data_yaml,
    epochs=100,
    batch=batch,          # A10G x4 → 64
    imgsz=1024,
    device=device,        # [0,1,2,3]
    workers=16,           # 48 vCPU
    patience=20,
    lr0=0.01,
    lrf=0.01,
    # 의료영상 augmentation
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.1,
    degrees=10.0, translate=0.1, scale=0.2,
    fliplr=0.5, flipud=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    # 출력
    project=TRAIN_DIR,
    name='vindr_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

elapsed = time.time() - start
print(f'\n학습 완료! ({elapsed/3600:.1f}시간)')

In [ ]:
# ============================================================
# Cell 4: 결과 확인 + S3 업로드
# ============================================================
import os, glob, shutil

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
REGION = 'ap-northeast-2'
TRAIN_DIR = '/home/ec2-user/SageMaker/vindr-cxr/yolov8_runs'
RUN_DIR = f'{TRAIN_DIR}/vindr_v1'

# best.pt 찾기
best_pt = f'{RUN_DIR}/weights/best.pt'
if not os.path.exists(best_pt):
    for pt in glob.glob(f'{TRAIN_DIR}/**/best.pt', recursive=True):
        best_pt = pt
        break

print(f'Best 모델: {best_pt}')
print(f'크기: {os.path.getsize(best_pt)/1e6:.1f}MB')

# 결과 파일 확인
print(f'\n결과 디렉토리: {RUN_DIR}')
if os.path.isdir(RUN_DIR):
    for f in sorted(os.listdir(RUN_DIR)):
        full = os.path.join(RUN_DIR, f)
        if os.path.isfile(full):
            print(f'  {f} ({os.path.getsize(full)/1e3:.0f}KB)')

# S3 업로드
print(f'\nS3 업로드...')
!aws s3 cp {best_pt} s3://{BUCKET}/models/yolov8_vindr_best.pt --region {REGION}
!aws s3 sync {RUN_DIR}/ s3://{BUCKET}/output/yolov8_vindr/ --region {REGION} --only-show-errors
print(f'\n완료!')
print(f'  모델: s3://{BUCKET}/models/yolov8_vindr_best.pt')
print(f'  결과: s3://{BUCKET}/output/yolov8_vindr/')